In [ ]:
import probablepeople as pp 
import os
import sys
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [ ]:
import os
import sys

# 1. Point directly to your installed JDK 17 path
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]

# Ensure PySpark uses the active Python interpreter
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 2. Now import and start Spark


spark = SparkSession.builder \
    .appName("pyspark_ylep") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "24") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.autoBroadcastJoinThreshold", "64MB") \
    .getOrCreate()
print("Spark initialized successfully!")

In [ ]:
df_read_business_data = spark.read.json("D:\\A_Data_engineer\\Project_Yelp\\Yelp JSON\\yelp_academic_dataset_business.json")

In [ ]:
invalid_placeholders=["", "N/A", "null", "None","Confidential"]

f_new_address_1 = df_read_business_data.withColumn( "new_address" , F.when( F.trim(F.col("address")).isin(invalid_placeholders)
                                                                             , "unknow address" ) .otherwise(F.regexp_replace(F.lower(F.col("address")) ,r"\bstreet\b","st") ) )


In [ ]:


# Dictionary of old_word: new_word
replacements = {"&": "and",
                ",": "",
                r"\bn\b\.\s*":"north",
                r"\bst\b\.s*":"saint",
                r"\bmt\b\.s*":"mount juliet",
                r"\bhts\b\.s*":"heights",
                r"\be\b\.s*":"east",
                r"\bw\b\.s*":"west",
                r"\bkop\b\s*":"king of prussia", 
                r"\btwp\b\s*":"township", 
                r"\bnj\b\s*":"new jersey", 
                r"\bfl\b\s*":"",
                r"\bphilly\b\s*":"philadelphia",
                r"\s*/.*$": "",
                r"-davidson metropolitan government \(balance\)": "",    
                r"\s*city \(balance\)": "",    
                r"\s*\(meridian\)": "", 
                r"\s*\(walsingham\)": "",
                r"\s*\(northeast philadelphia\)": "",
                }

# Chain regexp_replace for each pair
expr = F.lower(F.col("city"))
for old, new in replacements.items():
    print(old ,"      ",new)
    expr = F.regexp_replace(expr, old, new)

df_new = df_read_business_data.withColumn("cleaned_text", expr)